<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/01_first_deep_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · The whole game: your first Deep Agent

You are going to build a research assistant that plans its own work, searches the web, takes
notes into a filesystem, and writes a report — in about ten lines of code.

Then, once it has run, we will take apart what made that possible. **That order is deliberate.**
This course starts with a whole working agent and unpacks it afterwards, rather than making you
sit through six concepts before anything runs.

**New in this lesson:** `create_deep_agent`, the built-in tool suite, tracing, LangSmith Studio.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-01-first-agent"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Give it a way to see the web

The LangSmith gateway exposes the model provider's own web search, so this costs you no extra
key and no extra service. One line.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL)

# Provider-native web search: the model runs the search itself and cites what it found.
probe = llm.bind_tools([{"type": "web_search"}]).invoke(
    "In one sentence: what is LangChain's Deep Agents library?"
)
print(probe.text)

## 2. Build the agent

`create_deep_agent` takes a model, some tools, and instructions. Everything else — planning,
files, delegation — comes with it.

Note how web search is handed over: as a **tool spec in `tools=`**, not as a pre-bound model.
Deep Agents adds its own tools to whatever you pass, so it needs to do the binding itself.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=MODEL,
    tools=[{"type": "web_search"}],  # a provider tool, passed as a spec
    system_prompt=(
        "You are a research assistant.\n"
        "Research the user's question thoroughly using web search.\n"
        "Keep working notes in files as you go.\n"
        "Always write your final report to report.md before you finish.\n"
        "Cite sources inline."
    ),
)

print(type(agent).__name__)

## 3. Run it

This takes a minute or two. It is doing real work: searching, reading, note-taking, writing.

While you wait, notice what you did *not* have to write — no loop, no "now decide what to do
next", no scratchpad management, no file handling.

In [ ]:
QUESTION = "What are the main differences between LangGraph and Deep Agents, and when would you pick each?"

result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})

print(result["messages"][-1].text)

## 4. Look at what it left behind

You never told it to create files. Look anyway.

In [ ]:
files = result.get("files", {})

if not files:
    print("No files this time — the agent judged the question small enough to answer directly.")
    print("Try a broader question (more sub-topics) and it will start taking notes.")
else:
    print("Files the agent created:\n")
    for path in files:
        print(f"  {path}")

    if "report.md" in files:
        print("\n" + "=" * 60)
        print(files["report.md"][:1500])

In [ ]:
# It also planned. Every todo below was written by the agent, not by you.
for todo in result.get("todos", []):
    print(f"[{todo.get('status', '?'):>11}] {todo.get('content', todo)}")

---

## 5. The reveal — what you actually got

You wrote a model, a prompt, and one tool. What ran was a **harness**: a set of behaviours
wrapped around the model that turn "answer this" into "work on this until it is done".

| What you saw | What provided it |
|---|---|
| A todo list it wrote and worked through | planning middleware |
| `report.md` and working notes | a **filesystem** — `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep` |
| Long research that did not blow up the context window | summarization middleware, plus offloading notes to files |
| The ability to hand off a chunk of work | the `task` tool and a built-in general-purpose subagent |

That bundle is what "Deep Agent" means. It is not a different framework from LangChain — it is
a well-chosen default stack, which you will see for yourself in lesson 09.

**The rest of Part 1 pulls one layer out of that box at a time:**

| Lesson | Pulls out |
|---|---|
| 02 | the filesystem — and where files actually live |
| 03 | tools, and why a docstring is prompt engineering |
| 04 | tools you do not own, via MCP |
| 05 | subagents and context isolation |
| 06 | middleware, retries, PII, and human approval |
| 07 | memory that survives the conversation |
| 08 | skills — procedures loaded on demand |
| 09 | the LangGraph machinery underneath all of it |

### 🧠 Checkpoint

Look at the todo list and the messages above. The agent did not answer immediately — it wrote a
plan first, then worked through it. Nothing in your ten lines of code said "make a plan".

Where did that behaviour come from, and why does it matter for a question that takes 30 steps
to answer?

<details><summary>Show answer</summary>

It came from the **harness**, not from your prompt and not from the model alone. Deep Agents
ships a planning tool, and the system prompt around it tells the model to use one for
multi-step work.

It matters because a model answering in one shot has to hold the entire task in its head at
once. A model that writes a plan can work on step 3 without re-deriving steps 1, 2, 4, and 5 —
the plan is external memory. On a 30-step task that is the difference between finishing and
drifting.

This is the recurring theme of the course: **the interesting engineering is in the harness
around the model, not in the model call.**

</details>

---

## 6. See it in Studio

LangSmith Studio is a visual debugger for agents. It runs against the agent you just built, in
this runtime.

In [ ]:
from langsmith_studio_nb import start_studio

# Serves the notebook variable named `agent` and prints a link.
session = start_studio()

Open the link. Send it a research question and watch it work.

Three things to find:

1. **The message list** — every model call and tool call, in order.
2. **The state inspector** — `files` and `todos` update live as the agent works.
3. **Thread history** — each conversation is a thread you can revisit and fork.

> 📸 **`01-studio-overview.png`** — LangSmith Studio open on the research agent mid-run, with the message list on the left and the state inspector on the right showing `files` and `todos` populated.
>
> *Caption:* Studio: the agent's messages on the left, its live state on the right.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/01-studio-overview.png`

## 7. See it in LangSmith

Studio shows you what the agent *is doing*. LangSmith shows you what it *did* — every run,
permanently, with timings and token counts.

Go to [smith.langchain.com](https://smith.langchain.com) and open the project
**`lcw-01-first-agent`**. Your run is at the top.

> 📸 **`01-langsmith-trace.png`** — A LangSmith trace for the research agent, run tree expanded to show nested model calls and web_search tool calls, with the token count and latency columns visible.
>
> *Caption:* The same run as a trace: every step, its tokens, and its latency.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/01-langsmith-trace.png`

In [ ]:
# The trace is also queryable from code — you will lean on this in Part 2.
from langsmith import Client

client = Client()
runs = list(client.list_runs(project_name="lcw-01-first-agent", is_root=True, limit=1))

if runs:
    run = runs[0]
    print(f"latency:      {run.latency:.1f}s" if run.latency else "latency: n/a")
    print(f"total tokens: {run.total_tokens}")
    print(f"url:          {run.url}")
else:
    print("No runs found yet — traces can take a few seconds to land. Re-run this cell.")

### 🧠 Checkpoint

Open your trace in LangSmith and answer from the trace, not from memory:

How many **model calls** did that one question cost, and which single step used the most tokens?

<details><summary>Show answer</summary>

Expect somewhere between 5 and 20 model calls for one research question — many more than the
one call people usually picture.

The heaviest step is almost always a model call that happens *after* several web searches have
come back, because every prior search result is still sitting in the message history and gets
re-sent with the next call.

That single observation motivates the next two lessons: **files exist so that bulky content can
leave the message history**, and **subagents exist so it never enters the parent's history at
all.** If you can read that from a trace, you can debug an agent.

</details>

### ✍️ Exercise

Make it yours:

1. Change `QUESTION` to something you actually want researched.
2. Tighten the system prompt — try requiring a specific report structure, a word limit, or a
   "list what you could not verify" section.
3. Re-run, then open the new trace and compare it against the first one. Did the number of
   steps change? The token count?

The point is not the output. It is noticing that a prompt change is visible in the trace as a
different *shape* of work.

<details><summary>Show a solution</summary>

```python
agent = create_deep_agent(
    model=MODEL,
    tools=[{"type": "web_search"}],
    system_prompt=(
        "You are a research assistant.\n"
        "Research the question thoroughly using web search.\n"
        "Keep working notes in files as you go.\n"
        "\n"
        "Write your final report to report.md with exactly these sections:\n"
        "  ## Answer      - under 150 words\n"
        "  ## Evidence    - bullet points, each with an inline citation\n"
        "  ## Unverified  - claims you could not confirm from a source\n"
        "\n"
        "Never state something as fact without a citation. If sources disagree, say so."
    ),
)

result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print(result["files"]["report.md"])
```

</details>

---

## 📌 Key takeaways

- A Deep Agent is a model, plus tools, plus a **harness** — and the harness is where the engineering is.
- Planning and file-writing are not hardcoded features; they emerge from giving the model tools and telling it to use them.
- The filesystem exists to keep bulky content **out of the message history**, which is what makes long tasks affordable.
- Every run is a trace, and the trace — not the final answer — is the primary debugging surface.
- One LangSmith key buys models, tracing, and Studio. There is no second credential in this course.
- `create_deep_agent` is not a separate framework. It is `create_agent` plus a default middleware stack, as lesson 09 shows.

---

## ➡️ Next

**[02 · Files as the agent's workspace](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/02_backends.ipynb)**

The agent wrote files. Where did they actually go — and what happens to them when the
conversation ends? That question turns out to be the most important design decision in an agent.